In [2]:
import sys
import os

# 1. Allows the notebook to find the 'mcp_server' package
sys.path.append(os.path.abspath(os.path.join("..")))

# 2. Allows 'core.py' to find 'config.py' and other files inside its own folder
sys.path.append(os.path.abspath(os.path.join("../mcp_server")))

# Now your imports will execute properly
from mcp_server.core import *

from dotenv import load_dotenv; load_dotenv()
from langchain.tools import tool
from langchain.agents import create_agent

In [3]:
tools = [tool(download_paper), tool(ingest_new_pdfs), tool(retrieve)]

In [4]:
from IPython.display import Markdown, display

def mprint(text: str) -> None:
    display(Markdown(text))

In [5]:
system_prompt = """You are a helpful AI assistant which help user to understand research papers.
You have handful tools which can donwload and process and get the relevant context from papers.

Things to consider -
1. If the paper is already downloaded dont donwload again and process it just simply say it's already donwloaded.
2. Give answer only from the given context if the No relevant context found then simply say you dont know.
"""

In [6]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=tools,
    # state_schema=PaperList,
    checkpointer=InMemorySaver(),
    system_prompt=system_prompt
)

In [7]:
config = {"configurable": {"thread_id": "3"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="download attention is all you need paper")]},
    config=config
)

Done
Done


In [13]:
response

{'messages': [HumanMessage(content='download attention is all you need paper', additional_kwargs={}, response_metadata={}, id='a8caef14-50c6-405f-8652-e176d04634a5'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'download_paper', 'arguments': '{"title": "Attention is all you need"}'}, '__gemini_function_call_thought_signatures__': {'call_203072': 'EnMKcQFpFH0T/nYwR2cALww4hZYy2zCxMpPozhJ158O+1TlIOu4wayHUAbtI5DT7RNHJ1MeH4PYcaTkyH9Q48EW39sfGReSe+2axPiYZDVyDlyevPX2B1mmdhVYKAQw1YIARkaFYy9UGhJl0h/CapWX/tIfL'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a185f8-fe62-71c3-93b4-86c4e5df48e3-0', tool_calls=[{'name': 'download_paper', 'args': {'title': 'Attention is all you need'}, 'id': 'call_203072', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 322, 'output_tokens': 20, 'total_tokens': 342, 'input_token_details': {'cache_read': 0

In [12]:
mprint(response["messages"][-1].content[0]["text"])

The paper "Attention is all you need" has been downloaded and successfully processed. How can I help you understand it?

In [14]:
response = await agent.ainvoke(
    {"messages": [HumanMessage(content="in the attention paper vaswani talk about BERT?")]},
    config=config
)

In [15]:
response

{'messages': [HumanMessage(content='download attention is all you need paper', additional_kwargs={}, response_metadata={}, id='a8caef14-50c6-405f-8652-e176d04634a5'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'download_paper', 'arguments': '{"title": "Attention is all you need"}'}, '__gemini_function_call_thought_signatures__': {'call_203072': 'EnMKcQFpFH0T/nYwR2cALww4hZYy2zCxMpPozhJ158O+1TlIOu4wayHUAbtI5DT7RNHJ1MeH4PYcaTkyH9Q48EW39sfGReSe+2axPiYZDVyDlyevPX2B1mmdhVYKAQw1YIARkaFYy9UGhJl0h/CapWX/tIfL'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a185f8-fe62-71c3-93b4-86c4e5df48e3-0', tool_calls=[{'name': 'download_paper', 'args': {'title': 'Attention is all you need'}, 'id': 'call_203072', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 322, 'output_tokens': 20, 'total_tokens': 342, 'input_token_details': {'cache_read': 0

In [16]:
mprint(response["messages"][-1].content[0]["text"])

No, the paper "Attention is All You Need" (by Vaswani et al., 2017) does not discuss BERT.

BERT was introduced later by Devlin et al. in 2018. While BERT is based on the Transformer architecture developed in the "Attention is All You Need" paper, it is a separate work. References to BERT in the provided documents appear in later papers that cite or build upon the original Transformer work.

In [17]:
response = await agent.ainvoke(
    {"messages": [HumanMessage(content="which method is proposed in paper for define positional encoding ?")]},
    config=config
)

In [18]:
mprint(response["messages"][-1].content[0]["text"])

In the "Attention is all you need" paper, the authors propose using **sine and cosine functions of different frequencies** for positional encoding.

The formulas used are:
*   $PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d_{model}})$
*   $PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d_{model}})$

Where $pos$ is the position and $i$ is the dimension.

The authors chose this sinusoidal approach because they hypothesized it would allow the model to easily learn to attend by relative positions, as for any fixed offset $k$, $PE_{pos+k}$ can be represented as a linear function of $PE_{pos}$. While they also experimented with learned positional embeddings and found the results to be nearly identical, they ultimately chose the sinusoidal version.